Contextual Transition Analysis: This notebook combines the FSM outputs with contextual, demographic, social, weather, and mobility information to investigate factors associated with behavioural state transitions.

Paths

In [ ]:
import pandas as pd

# Load shared project paths and configuration
%run 00_config.ipynb

# =========================
# INPUT FILES
# =========================

FSM_FILE = OUTPUT_DIR / "fsm_simulation_all_users.xlsx"

WEATHER_FILE = DATA_FEB_DIR / "Weather_file.csv"

PARTICIPANTS_FILE = (
    DATA_FEB_DIR
    / "participants-file-com-bmi.xlsx"
)

SOCIAL_FILE = SAMSUNG_DIR / "Social_file.csv"

ROUTES_FILE = (
    SAMSUNG_DIR
    / "rotas_features_final.xlsx"
)

Load datasets

In [ ]:
fsm_results = pd.read_excel(FSM_FILE)
weather = pd.read_csv(WEATHER_FILE)
participants = pd.read_excel(PARTICIPANTS_FILE)
social = pd.read_csv(SOCIAL_FILE)
routes = pd.read_excel(ROUTES_FILE)

Create binary feature for transition

In [ ]:
transition_df = fsm_results[
    fsm_results["previous_state"] != "None (Onboarding Init)"
].copy()

# Create binary transition variable
# 1 = transition between different states
# 0 = state maintenance
transition_df["transition"] = (
    transition_df["previous_state"]
    != transition_df["new_state"]
).astype(int)

Transition distribution

In [ ]:
n_records = len(transition_df)
n_transitions = transition_df["transition"].sum()
n_non_transitions = n_records - n_transitions

transition_summary = pd.DataFrame({
    "metric": [
        "Total analysed records",
        "Transition days",
        "Non-transition days",
        "Percentage of transition days",
        "Percentage of non-transition days"
    ],
    "value": [
        n_records,
        n_transitions,
        n_non_transitions,
        round(n_transitions / n_records * 100, 2),
        round(n_non_transitions / n_records * 100, 2)
    ]
})

transition_summary

Prepare Weather file

In [ ]:
weather["date"] = pd.to_datetime(
    weather["timestamp"]
).dt.date

weather_daily = (
    weather
    .groupby("date")
    .agg(
        temperature_2m=("temperature_2m", "mean"),
        precipitation=("precipitation", "sum")
    )
    .reset_index()
)

weather_daily["rain_day"] = (
    weather_daily["precipitation"] > 1
).astype(int)

Prepare Social Context

In [ ]:
social["date"] = pd.to_datetime(
    social["timestamp"]
).dt.date

social["social_context"] = pd.to_numeric(
    social["social_context"],
    errors="coerce"
)

# Keep only valid social context responses for daily aggregation.
# Days without social data remain NaN after the merge.
social_valid = social.dropna(
    subset=["social_context"]
).copy()

social_valid["social_context"] = (
    social_valid["social_context"]
    .astype(int)
)

# Social context is categorical, so the daily mode is used.
social_daily = (
    social_valid
    .groupby(["user_id", "date"])
    .agg(
        social_context=("social_context", lambda x: x.mode().iloc[0])
    )
    .reset_index()
)

# Social context coding:
# 0 = alone
# 1 = friends
# 2 = family
# 3 = peers/co-workers
# 4 = others

social_daily["social_presence"] = (
    social_daily["social_context"] != 0
).astype(int)

Prepare participants file

In [ ]:
participants_selected = participants[
    [
        "user_id",
        "gender",
        "bmi_class"
    ]
].copy()

def classify_bmi_profile(bmi_class):
    bmi_class = str(bmi_class).lower()

    if (
        "obes" in bmi_class
        or "excesso" in bmi_class
        or "overweight" in bmi_class
    ):
        return "overweight_obese"

    return "healthy"

participants_selected["bmi_profile"] = (
    participants_selected["bmi_class"]
    .apply(classify_bmi_profile)
)

participants_selected = participants_selected[
    [
        "user_id",
        "gender",
        "bmi_profile"
    ]
]

Prepare Routes file

In [ ]:
routes["date"] = pd.to_datetime(
    routes["day"]
).dt.date

routes["has_space_transition"] = (
    routes["space_transition"]
    .astype(str)
    .str.lower()
    != "same"
).astype(int)

routes_daily = (
    routes
    .groupby(["user_id", "date"])
    .agg(
        n_routes=("route_id", "count"),
        mean_route_duration=("route_duration_min", "mean"),
        n_space_transitions=("has_space_transition", "sum")
    )
    .reset_index()
)

Merge contextual variables

In [ ]:
context_df = transition_df[
    [
        "user_id",
        "date",
        "previous_state",
        "new_state",
        "transition"
    ]
].copy()

context_df["date"] = pd.to_datetime(
    context_df["date"]
).dt.date

context_df = context_df.merge(
    participants_selected,
    on="user_id",
    how="left"
)

context_df = context_df.merge(
    weather_daily,
    on="date",
    how="left"
)

context_df = context_df.merge(
    social_daily,
    on=["user_id", "date"],
    how="left"
)

context_df = context_df.merge(
    routes_daily,
    on=["user_id", "date"],
    how="left"
)

context_df.head()

Handle days without routes

In [ ]:
context_df["n_routes"] = context_df["n_routes"].fillna(0)
context_df["mean_route_duration"] = context_df["mean_route_duration"].fillna(0)
context_df["n_space_transitions"] = context_df["n_space_transitions"].fillna(0)

# Weekend is calculated directly from the FSM date, so it is available even when no routes were recorded.
context_df["is_weekend"] = (
    pd.to_datetime(context_df["date"])
    .dt.dayofweek
    .isin([5, 6])
    .astype(int)
)

Select final variables

In [ ]:
selected_columns = [
    "user_id",
    "date",
    "previous_state",
    "new_state",
    "transition",
    "gender",
    "bmi_profile",
    "temperature_2m",
    "precipitation",
    "rain_day",
    "social_context",
    "social_presence",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions",
    "is_weekend"
]


Check final dataset

In [ ]:
print("Final dataset shape:", context_df.shape)

missing_summary = (
    context_df
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = [
    "variable",
    "missing_values"
]

missing_summary

Save dataset

In [ ]:
CONTEXT_DATASET_FILE = OUTPUT_DIR / "contextual_transition_dataset.xlsx"

context_df.to_excel(
    CONTEXT_DATASET_FILE,
    index=False
)

print("Saved contextual transition dataset.")

Define variables

In [ ]:
continuous_variables = [
    "temperature_2m",
    "precipitation",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions"
]

descriptive statistics for continuous variables

In [ ]:
continuous_descriptive = (
    context_df
    .groupby("transition")[continuous_variables]
    .agg(["count", "mean", "median", "std"])
)

continuous_descriptive

Normality tests

In [ ]:
from scipy.stats import shapiro, levene

print("=== STRUCTURAL ASSUMPTIONS CHECK ===")

for variable in continuous_variables:
    temp_df = context_df[
        ["transition", variable]
    ].dropna()

    no_transition = temp_df[
        temp_df["transition"] == 0
    ][variable]

    transition = temp_df[
        temp_df["transition"] == 1
    ][variable]

    # Test normality of the variable distribution
    _, p_shapiro = shapiro(temp_df[variable])

    # Test homogeneity of variances between groups
    p_levene = None

    if len(no_transition) > 1 and len(transition) > 1:
        _, p_levene = levene(
            no_transition,
            transition
        )

    print(f"\nVariable: {variable}")
    print(
        f"Shapiro-Wilk p-value: {p_shapiro:.5f} | "
        f"{'Non-normal' if p_shapiro < 0.05 else 'Normal'}"
    )

    if p_levene is not None:
        print(
            f"Levene p-value: {p_levene:.5f} | "
            f"{'Unequal variances' if p_levene < 0.05 else 'Equal variances'}"
        )

Mann-Whitney Tests

In [ ]:
from scipy.stats import mannwhitneyu

mann_whitney_results = []

for variable in continuous_variables:

    temp_df = context_df[
        ["transition", variable]
    ].dropna()

    no_transition = temp_df[
        temp_df["transition"] == 0
    ][variable]

    transition = temp_df[
        temp_df["transition"] == 1
    ][variable]

    statistic, p_value = mannwhitneyu(
        no_transition,
        transition,
        alternative="two-sided"
    )

    mann_whitney_results.append({
        "variable": variable,
        "median_no_transition": no_transition.median(),
        "median_transition": transition.median(),
        "p_value": p_value
    })

mann_whitney_results = pd.DataFrame(
    mann_whitney_results
)

mann_whitney_results

Categorical contigency tables

In [ ]:
categorical_variables = [
    "gender",
    "bmi_profile",
    "rain_day",
    "is_weekend",
    "social_presence",
    "social_context"
]

for variable in categorical_variables:
    print("\n" + "=" * 60)
    print(variable)
    print("=" * 60)

    print(
        pd.crosstab(
            context_df[variable],
            context_df["transition"]
        )
    )

Fisher Exact Tests

In [ ]:
from scipy.stats import fisher_exact

fisher_variables = [
    "gender",
    "bmi_profile",
    "rain_day",
    "is_weekend",
    "social_presence"
]

fisher_results = []

for variable in fisher_variables:

    temp_df = context_df[
        [variable, "transition"]
    ].dropna()

    contingency_table = pd.crosstab(
        temp_df[variable],
        temp_df["transition"]
    )

    odds_ratio, p_value = fisher_exact(
        contingency_table
    )

    fisher_results.append({
        "variable": variable,
        "odds_ratio": odds_ratio,
        "p_value": p_value
    })

fisher_results = pd.DataFrame(
    fisher_results
)

fisher_results

Contigency table for social context

In [ ]:
from scipy.stats import chi2_contingency

social_df = context_df[
    ["social_context", "transition"]
].dropna()

social_contingency = pd.crosstab(
    social_df["social_context"],
    social_df["transition"]
)

social_contingency

Chi-Square test

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(
    social_contingency
)

chi_square_results = pd.DataFrame({
    "chi2": [chi2],
    "degrees_of_freedom": [dof],
    "p_value": [p_value]
})

chi_square_results

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=social_contingency.index,
    columns=social_contingency.columns
)

expected_df

Save univariate statistical results

In [ ]:
mann_whitney_results.to_excel(
    RESULTS_DIR / "07a_mann_whitney_general_transitions.xlsx",
    index=False
)

fisher_results.to_excel(
    RESULTS_DIR / "07b_fisher_general_transitions.xlsx",
    index=False
)

chi_square_results.to_excel(
    RESULTS_DIR / "07c_chisquare_social_context.xlsx",
    index=False
)

print(
    "Saved all general univariate statistical results "
    "(07a, 07b, 07c) to Excel."
)

Random Forest for feature importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# Keep rows with available social and weather information
df_ml = context_df.dropna(
    subset=["social_context", "temperature_2m"]
).copy()

features_list = [
    "gender",
    "bmi_profile",
    "temperature_2m",
    "precipitation",
    "rain_day",
    "is_weekend",
    "social_presence",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions"
]

X = df_ml[features_list].copy()
X = pd.get_dummies(X, drop_first=True)

y = df_ml["transition"]

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="f1_weighted"
)

print(
    f"Weighted F1-score (5-fold CV): "
    f"{np.mean(cv_scores) * 100:.2f}% "
    f"(+/- {np.std(cv_scores) * 100:.2f}%)"
)

rf_model.fit(X, y)

importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances_df = importances.reset_index()
importances_df.columns = [
    "feature",
    "importance"
]

importances_df

Random Forest without social variables

In [ ]:
# Random Forest without social variables

df_ml_no_social = context_df.dropna(
    subset=[
        "temperature_2m",
        "precipitation",
        "rain_day"
    ]
).copy()

features_no_social = [
    "gender",
    "bmi_profile",
    "temperature_2m",
    "precipitation",
    "rain_day",
    "is_weekend",
    "n_routes",
    "mean_route_duration",
    "n_space_transitions"
]

X_no_social = df_ml_no_social[
    features_no_social
].copy()

X_no_social = pd.get_dummies(
    X_no_social,
    drop_first=True
)

y_no_social = df_ml_no_social["transition"]

rf_no_social = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

cv_scores_no_social = cross_val_score(
    rf_no_social,
    X_no_social,
    y_no_social,
    cv=cv,
    scoring="f1_weighted"
)

print(
    f"Weighted F1-score without social variables: "
    f"{np.mean(cv_scores_no_social) * 100:.2f}% "
    f"(+/- {np.std(cv_scores_no_social) * 100:.2f}%)"
)

rf_no_social.fit(
    X_no_social,
    y_no_social
)

importances_no_social = pd.Series(
    rf_no_social.feature_importances_,
    index=X_no_social.columns
).sort_values(ascending=False)

importances_no_social_df = (
    importances_no_social
    .reset_index()
)

importances_no_social_df.columns = [
    "feature",
    "importance"
]

importances_no_social_df

Save

In [ ]:
importances_no_social_df.to_excel(
    RESULTS_DIR / "09_random_forest_general_importances_no_social.xlsx",
    index=False
)